# Decoding hyperparameter sweep

Compare decoding performance across `n_voxels` and `lambda` for each ROI.
Separate panels for gabor orientation and abstract value decoding.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from abstract_values.utils.data import BIDS_FOLDER

sns.set_theme(style='ticks', font_scale=1.1)

In [ ]:
DERIV_DIR = BIDS_FOLDER / 'derivatives' / 'decoding'

# Auto-discover subjects
SUBJECTS = sorted({
    d.name.removeprefix('sub-')
    for kind in ['gabor', 'value']
    if (DERIV_DIR / kind).exists()
    for d in (DERIV_DIR / kind).iterdir()
    if d.is_dir() and d.name.startswith('sub-')
})
print(f'Subjects: {SUBJECTS}')

ROIS           = ['BensonV1', 'NPCr']
N_VOXELS_GRID  = [0, 50, 100, 250, 500]
LAMBDA_GRID    = [0.0, 0.1]
SMOOTHED_GRID  = [False, True]
NOISE          = 'full'

In [ ]:
def get_mapping(subject, session):
    num = int(''.join(c for c in str(subject) if c.isdigit()))
    if num % 2 == 0:
        return 'cdf' if session == 1 else 'inverse_cdf'
    return 'inverse_cdf' if session == 1 else 'cdf'


def load_decoding(kind, subject, mask, n_voxels=100, noise=NOISE,
                  smoothed=SMOOTHED, lambd=0.0):
    """Load decoding posterior TSV; returns None if not found."""
    smooth_label = '_smoothed' if smoothed else ''
    lambd_label  = f'_lambda-{lambd}' if lambd != 0.0 else ''
    sub_dir = DERIV_DIR / kind / f'sub-{subject}'
    if not sub_dir.exists():
        return None

    def _load(fn):
        df = pd.read_csv(fn, sep='\t', index_col=list(range(4)))
        df.columns = df.columns.astype(float)
        return df

    dfs = []
    for sd in sorted(sub_dir.iterdir()):
        if not sd.is_dir():
            continue
        fn = (sd / 'func' /
              f'sub-{subject}_{sd.name}_mask-{mask}'
              f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_pars.tsv')
        if fn.exists():
            dfs.append(_load(fn))
    fn_all = (sub_dir / 'func' /
              f'sub-{subject}_mask-{mask}'
              f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_pars.tsv')
    if fn_all.exists():
        dfs.append(_load(fn_all))
    if not dfs:
        return None
    combined = pd.concat(dfs).sort_index()
    return combined[~combined.index.duplicated(keep='first')]


def load_meta(kind, subject, mask, n_voxels=100, noise=NOISE,
              smoothed=SMOOTHED, lambd=0.0):
    """Load fold metadata TSV (n_voxels_selected per fold)."""
    smooth_label = '_smoothed' if smoothed else ''
    lambd_label  = f'_lambda-{lambd}' if lambd != 0.0 else ''
    sub_dir = DERIV_DIR / kind / f'sub-{subject}'
    if not sub_dir.exists():
        return None

    # Try all-sessions path first
    fn = (sub_dir / 'func' /
          f'sub-{subject}_mask-{mask}'
          f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_meta.tsv')
    if fn.exists():
        return pd.read_csv(fn, sep='\t')

    # Try per-session paths
    for sd in sorted(sub_dir.iterdir()):
        if not sd.is_dir():
            continue
        fn = (sd / 'func' /
              f'sub-{subject}_{sd.name}_mask-{mask}'
              f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_meta.tsv')
        if fn.exists():
            return pd.read_csv(fn, sep='\t')
    return None


def normalize_rows(df):
    return df.div(df.sum(axis=1), axis=0)


def posterior_mean_orientation(df):
    cols = df.columns.values.astype(float)
    norm = normalize_rows(df).values
    return np.arctan2(norm @ np.sin(2 * cols), norm @ np.cos(2 * cols)) / 2 % np.pi


def posterior_mean_value(df):
    return normalize_rows(df).values @ df.columns.values.astype(float)


def circular_correlation(true_rad, pred_rad):
    t2, p2 = 2*true_rad, 2*pred_rad
    return float((np.corrcoef(np.cos(t2), np.cos(p2))[0,1] +
                  np.corrcoef(np.sin(t2), np.sin(p2))[0,1]) / 2)

## Build sweep DataFrame

In [ ]:
rows = []

for subject in SUBJECTS:
    for roi in ROIS:
        for nv in N_VOXELS_GRID:
            for lam in LAMBDA_GRID:
                for smoothed in SMOOTHED_GRID:
                    for kind in ['gabor', 'value']:
                        df = load_decoding(kind, subject, roi, n_voxels=nv,
                                           lambd=lam, smoothed=smoothed)
                        if df is None:
                            continue

                        # Get n_voxels actually used (from meta if available)
                        meta = load_meta(kind, subject, roi, n_voxels=nv,
                                         lambd=lam, smoothed=smoothed)
                        if meta is not None:
                            mean_nvox = meta['n_voxels_selected'].mean()
                        else:
                            mean_nvox = nv if nv > 0 else np.nan

                        if kind == 'gabor':
                            true_v = df.index.get_level_values('true_orientation_rad').astype(float)
                            pred_v = posterior_mean_orientation(df)
                            acc = circular_correlation(true_v, pred_v)
                            acc_label = 'circ_r'
                        else:
                            true_v = df.index.get_level_values('true_value_chf').astype(float)
                            pred_v = posterior_mean_value(df)
                            acc, _ = pearsonr(true_v, pred_v)
                            acc_label = 'pearson_r'

                        rows.append(dict(
                            subject=subject, roi=roi, kind=kind,
                            n_voxels=nv, lambd=lam,
                            smoothed='smoothed' if smoothed else 'unsmoothed',
                            n_voxels_actual=mean_nvox,
                            accuracy=acc, acc_metric=acc_label,
                        ))

sweep = pd.DataFrame(rows)
print(f'{len(sweep)} rows  |  subjects: {sweep["subject"].nunique()}  |  '
      f'combos with data: {len(sweep.groupby(["roi","n_voxels","lambd","kind","smoothed"]))}')
sweep.head(10)

## Gabor orientation decoding

In [ ]:
gabor = sweep[sweep['kind'] == 'gabor'].copy()
gabor['n_voxels_label'] = gabor['n_voxels'].map(lambda x: 'CV' if x == 0 else str(x))
gabor['lambda'] = gabor['lambd'].astype(str)

if not gabor.empty:
    order = ['CV'] + [str(x) for x in N_VOXELS_GRID if x > 0]
    g = sns.catplot(
        data=gabor, x='n_voxels_label', y='accuracy',
        hue='lambda', col='roi', row='smoothed',
        kind='point', dodge=0.15, markers='o', linestyles='-',
        errorbar=('se', 1), capsize=0.1, err_kws={'linewidth': 1.5},
        palette={'0.0': 'tomato', '0.1': 'steelblue'},
        col_order=[r for r in ROIS if r in gabor['roi'].unique()],
        row_order=['unsmoothed', 'smoothed'],
        height=4, aspect=1.2,
        order=order,
    )
    for ax in g.axes.flat:
        ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
        ax.set_xlabel('n_voxels')
        ax.set_ylabel('Circular correlation (r)')
    g.figure.suptitle('Gabor decoding: circular correlation by n_voxels and \u03bb',
                      y=1.02, fontsize=13)
    plt.show()
else:
    print('No gabor sweep data.')

## Abstract value decoding

In [ ]:
value = sweep[sweep['kind'] == 'value'].copy()
value['n_voxels_label'] = value['n_voxels'].map(lambda x: 'CV' if x == 0 else str(x))
value['lambda'] = value['lambd'].astype(str)

if not value.empty:
    order = ['CV'] + [str(x) for x in N_VOXELS_GRID if x > 0]
    g = sns.catplot(
        data=value, x='n_voxels_label', y='accuracy',
        hue='lambda', col='roi', row='smoothed',
        kind='point', dodge=0.15, markers='o', linestyles='-',
        errorbar=('se', 1), capsize=0.1, err_kws={'linewidth': 1.5},
        palette={'0.0': 'tomato', '0.1': 'steelblue'},
        col_order=[r for r in ROIS if r in value['roi'].unique()],
        row_order=['unsmoothed', 'smoothed'],
        height=4, aspect=1.2,
        order=order,
    )
    for ax in g.axes.flat:
        ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
        ax.set_xlabel('n_voxels')
        ax.set_ylabel('Pearson r')
    g.figure.suptitle('Value decoding: Pearson r by n_voxels and \u03bb',
                      y=1.02, fontsize=13)
    plt.show()
else:
    print('No value sweep data.')

## Number of voxels selected (n_voxels=0 / CV mode)

In [ ]:
cv_rows = sweep[(sweep['n_voxels'] == 0) & sweep['n_voxels_actual'].notna()].copy()

if not cv_rows.empty:
    g = sns.catplot(data=cv_rows, x='roi', y='n_voxels_actual',
                    hue='kind', col='smoothed',
                    kind='swarm', dodge=True, size=6,
                    palette={'gabor': 'teal', 'value': 'coral'},
                    col_order=['unsmoothed', 'smoothed'],
                    height=4, aspect=1.0)
    for ax in g.axes.flat:
        ax.set_xlabel('')
        ax.set_ylabel('n voxels selected (mean across folds)')
    g.figure.suptitle('Voxels with nested CV R\u00b2 > 0', y=1.02, fontsize=13)
    plt.show()

    print(cv_rows[['subject', 'roi', 'kind', 'lambd', 'smoothed', 'n_voxels_actual']]
          .to_string(index=False))
else:
    print('No n_voxels=0 results with metadata yet.')

## Summary table

In [ ]:
if not sweep.empty:
    summary = (sweep
               .groupby(['kind', 'roi', 'n_voxels', 'lambd'])
               .agg(mean_acc=('accuracy', 'mean'),
                    std_acc=('accuracy', 'std'),
                    n_subjects=('subject', 'nunique'))
               .round(3)
               .reset_index())
    summary['n_voxels_label'] = summary['n_voxels'].map(lambda x: 'CV' if x == 0 else str(x))
    print(summary.to_string(index=False))
else:
    print('No data yet.')

## Missing data

Which subject / ROI / n_voxels / lambda / smoothed / kind combinations have no results yet?

In [ ]:
# Build the full expected grid and compare to what we have
from itertools import product

expected = pd.DataFrame(list(product(
    SUBJECTS, ROIS, N_VOXELS_GRID, LAMBDA_GRID,
    ['unsmoothed', 'smoothed'], ['gabor', 'value']
)), columns=['subject', 'roi', 'n_voxels', 'lambd', 'smoothed', 'kind'])

if not sweep.empty:
    merged = expected.merge(
        sweep[['subject', 'roi', 'n_voxels', 'lambd', 'smoothed', 'kind', 'accuracy']],
        how='left', on=['subject', 'roi', 'n_voxels', 'lambd', 'smoothed', 'kind'])
    missing = merged[merged['accuracy'].isna()].drop(columns='accuracy')

    n_total = len(expected)
    n_have = n_total - len(missing)
    print(f'{n_have}/{n_total} cells have data ({len(missing)} missing)\n')

    if not missing.empty:
        # Summarise: per subject, count missing
        summary = (missing
                   .groupby(['subject', 'smoothed'])
                   .size()
                   .reset_index(name='n_missing'))
        print(summary.to_string(index=False))
        print()
        print(missing.to_string(index=False))
    else:
        print('All cells have data!')
else:
    print(f'No data at all. {len(expected)} cells expected.')